# 01 – Datensammlung
**Projekt Gruppe 37 – Data Analytics HS2026 (ZHAW)**
*Marc Stump, Ayushan Ahilan*

**Forschungsfrage:** Welche Faktoren beeinflussen, ob ein Zug mit ≥ 3 Minuten Verspätung ankommt – und unterscheiden sich diese Faktoren zwischen **Deutschschweiz und Romandie** sowie zwischen **Stosszeiten und Randzeiten**?

| # | Quelle | Zugriff | Inhalt |
|---|---|---|---|
| 1 | Ist-Daten v2, opentransportdata.swiss | CKAN-Web-API → Tagesdateien (CSV) | Soll- und Ist-Zeiten aller ÖV-Halte |
| 2 | Service Points (atlas), opentransportdata.swiss | CKAN-Web-API → CSV | Koordinaten, Kanton, Gemeinde jeder Haltestelle |
| 3 | Open-Meteo Historical Weather API | REST-API (JSON) | stündliches Wetter pro Bahnhof |

Die Daten werden **von uns selbst über Web-APIs abgefragt, gefiltert und in SQLite gespeichert** – es wird kein fertiger Analyse-Datensatz verwendet.

> Die vollständige Sammlung dauert je nach Internet ca. 30–90 Minuten (14 × ~500 MB). Empfohlen: im Terminal `python run_collection.py` ausführen. Die Zellen unten zeigen die API-Aufrufe Schritt für Schritt.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.db import get_connection

pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
config.OUTPUT_DIR.mkdir(exist_ok=True)
conn = get_connection()
print("Datenbank:", config.DB_PATH)

## 1. Ist-Daten: verfügbare Tage über die CKAN-API abfragen
Die Plattform publiziert pro Betriebstag eine CSV-Datei. Über die Methode `package_show` der CKAN-API erhalten wir die Liste aller Dateien inkl. Download-URL.

In [ ]:
from src import collect_istdaten
from src.http_utils import get_json

try:
    antwort = get_json(f"{config.CKAN_BASES[0]}/package_show", params={"id": config.ISTDATEN_PACKAGE})
    ressourcen = antwort["result"]["resources"]
    print("Anzahl Ressourcen:", len(ressourcen))
    print("Beispiel-URL:", ressourcen[-1].get("url"))
    verfuegbar = collect_istdaten.list_available_days()
    print(f"{len(verfuegbar)} Tage verfügbar: {min(verfuegbar)} bis {max(verfuegbar)}")
    print("Für die Analyse ausgewählt:", [str(d) for d in collect_istdaten.choose_days(verfuegbar)])
except Exception as e:
    print("API nicht erreichbar:", e)

## 2. Wetter: Beispielabfrage Open-Meteo (Zürich HB)

In [ ]:
try:
    beispiel = get_json(config.OPEN_METEO_URL, params={
        "latitude": 47.3782, "longitude": 8.5402,
        "start_date": "2026-10-01", "end_date": "2026-10-01",
        "hourly": "temperature_2m,precipitation,snowfall,wind_speed_10m",
        "timezone": "Europe/Zurich"})
    display(pd.DataFrame(beispiel["hourly"]).head(10))
except Exception as e:
    print("API nicht erreichbar:", e)

## 3. Vollständige Datensammlung
Setze `RUN_COLLECTION = True`, um alles direkt aus dem Notebook zu laden (alternativ `python run_collection.py` im Terminal). Bereits geladene Tage werden übersprungen.

Ablauf:
1. **Stationen** – Service-Point-CSV laden, Schweizer Haltestellen mit Koordinaten behalten, Sprachregion zuordnen
2. **Ist-Daten** – pro Tag herunterladen, in Chunks filtern (nur Züge, keine Durchfahrten, nur Schweizer Halte), Verspätung = Ist − Soll berechnen, in Tabelle `halte` speichern, Rohdatei löschen
3. **Wetter** – für die 100 meistfrequentierten Bahnhöfe stündliche Wetterdaten abfragen

In [ ]:
RUN_COLLECTION = False

if RUN_COLLECTION:
    from src import collect_stations, collect_weather
    collect_stations.collect(conn)
    collect_istdaten.collect(conn, n_days=config.N_DAYS)
    collect_weather.collect(conn, n=config.TOP_N_STATIONS)

## 4. Kontrolle der gesammelten Daten (SQL-Abfragen aus Python)

In [ ]:
display(pd.read_sql("SELECT * FROM sammel_log ORDER BY quelle, schluessel", conn))
display(pd.read_sql("""
    SELECT 'stationen' AS tabelle, COUNT(*) AS zeilen FROM stationen
    UNION ALL SELECT 'halte', COUNT(*) FROM halte
    UNION ALL SELECT 'wetter', COUNT(*) FROM wetter""", conn))

In [ ]:
display(pd.read_sql("SELECT * FROM halte LIMIT 5", conn))
display(pd.read_sql("SELECT * FROM stationen LIMIT 5", conn))
display(pd.read_sql("SELECT * FROM wetter LIMIT 5", conn))